# Experiment 1: Data Collection

**Objective:** Build a Python data-collection pipeline that gathers weather and air-quality data for selected Bengaluru locations and scrapes book details from a practice bookstore website.

The collected data is saved as CSV files so it can be used later for cleaning, visualization, and modelling.

## Plan

The experiment is split into three parts:

1. Weather data for five Bengaluru locations
2. Air-quality data for the same locations
3. Book title, price, and rating from a paginated practice website

The API sections use Open-Meteo. The scraping section uses Books to Scrape.

In [ ]:
from pathlib import Path
import time

import pandas as pd
import requests
from bs4 import BeautifulSoup

OUTPUT_DIR = Path(".")
OUTPUT_DIR.mkdir(exist_ok=True)

locations = [
    {"name": "MG Road", "lat": 12.9757, "lon": 77.6011},
    {"name": "Whitefield", "lat": 12.9698, "lon": 77.7500},
    {"name": "Jayanagar", "lat": 12.9308, "lon": 77.5838},
    {"name": "Hebbal", "lat": 13.0350, "lon": 77.5970},
    {"name": "Electronic City", "lat": 12.8399, "lon": 77.6770},
]

WEATHER_URL = "https://api.open-meteo.com/v1/forecast"
AIR_QUALITY_URL = "https://air-quality-api.open-meteo.com/v1/air-quality"

session = requests.Session()
session.headers.update({"User-Agent": "data-science-lab/1.0"})

print("Setup complete.")
print("Locations:", ", ".join(place["name"] for place in locations))

## Algorithm

1. Define the five Bengaluru locations with latitude and longitude.
2. Request hourly weather data and keep the latest 21 complete days.
3. Request hourly air-quality data and keep the latest 30 complete days.
4. Scrape each catalogue page from Books to Scrape.
5. Store title, price, and rating for every book.
6. Save the three datasets as CSV files and check their sizes.

## 1. Weather data

For each location, the notebook requests hourly temperature and relative humidity. Only the last 21 complete days are kept, so every location contributes the same number of hourly records.

In [ ]:
weather_frames = []

for place in locations:
    params = {
        "latitude": place["lat"],
        "longitude": place["lon"],
        "hourly": "temperature_2m,relative_humidity_2m",
        "timezone": "Asia/Kolkata",
        "past_days": 21,
        "forecast_days": 1,
    }

    response = session.get(WEATHER_URL, params=params, timeout=30)
    response.raise_for_status()
    hourly = response.json()["hourly"]

    df = pd.DataFrame(hourly).rename(
        columns={
            "temperature_2m": "temperature_c",
            "relative_humidity_2m": "humidity_percent",
        }
    )

    df["time"] = pd.to_datetime(df["time"])

    # Keep completed days only. The tail makes the result exactly 21 x 24 rows.
    today = pd.Timestamp.now(tz="Asia/Kolkata").tz_localize(None).normalize()
    df = df[df["time"] < today].tail(21 * 24).copy()

    df.insert(0, "location", place["name"])
    df.insert(1, "latitude", place["lat"])
    df.insert(2, "longitude", place["lon"])

    weather_frames.append(df)
    print(f'{place["name"]}: {len(df)} weather rows')

    time.sleep(0.25)

weather_df = pd.concat(weather_frames, ignore_index=True)

weather_file = OUTPUT_DIR / "bengaluru_weather.csv"
weather_df.to_csv(weather_file, index=False)

print("\nSaved:", weather_file)
print("Shape:", weather_df.shape)
weather_df.head()

## 2. Air-quality data

The same five coordinates are used for PM10, PM2.5, and carbon monoxide. This section keeps 30 complete days of hourly readings.

In [ ]:
air_frames = []

for place in locations:
    params = {
        "latitude": place["lat"],
        "longitude": place["lon"],
        "hourly": "pm10,pm2_5,carbon_monoxide",
        "timezone": "Asia/Kolkata",
        "past_days": 30,
        "forecast_days": 1,
    }

    response = session.get(AIR_QUALITY_URL, params=params, timeout=30)
    response.raise_for_status()
    hourly = response.json()["hourly"]

    df = pd.DataFrame(hourly)
    df["time"] = pd.to_datetime(df["time"])

    today = pd.Timestamp.now(tz="Asia/Kolkata").tz_localize(None).normalize()
    df = df[df["time"] < today].tail(30 * 24).copy()

    df.insert(0, "location", place["name"])
    df.insert(1, "latitude", place["lat"])
    df.insert(2, "longitude", place["lon"])

    air_frames.append(df)
    print(f'{place["name"]}: {len(df)} air-quality rows')

    time.sleep(0.25)

air_quality_df = pd.concat(air_frames, ignore_index=True)

air_file = OUTPUT_DIR / "bengaluru_air_quality.csv"
air_quality_df.to_csv(air_file, index=False)

print("\nSaved:", air_file)
print("Shape:", air_quality_df.shape)
air_quality_df.head()

In [ ]:
print("Weather date range:")
print(weather_df["time"].min(), "to", weather_df["time"].max())

print("\nAir-quality date range:")
print(air_quality_df["time"].min(), "to", air_quality_df["time"].max())

## 3. Bookstore web scraping

Books to Scrape is a practice website made for scraping exercises. The loop follows the **Next** link until the last catalogue page and collects each book's title, price, and star rating.

In [ ]:
BOOKS_BASE = "https://books.toscrape.com/catalogue/"
next_page = "page-1.html"

rating_value = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5,
}

books = []
page_number = 0

while next_page:
    page_number += 1
    response = session.get(BOOKS_BASE + next_page, timeout=30)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    for card in soup.select("article.product_pod"):
        title = card.select_one("h3 a")["title"]

        price_text = card.select_one(".price_color").get_text(strip=True)
        price_gbp = float(price_text.replace("£", "").replace("Â", ""))

        rating_class = card.select_one("p.star-rating").get("class", [])
        rating_word = next(
            name for name in rating_class if name != "star-rating"
        )

        books.append(
            {
                "title": title,
                "price_gbp": price_gbp,
                "rating": rating_value[rating_word],
            }
        )

    next_link = soup.select_one("li.next a")
    next_page = next_link["href"] if next_link else None

    print(f"Page {page_number}: {len(books)} books collected", end="\r")
    time.sleep(0.2)

books_df = pd.DataFrame(books)

books_file = OUTPUT_DIR / "books_data.csv"
books_df.to_csv(books_file, index=False)

print(f"\nSaved: {books_file}")
print("Shape:", books_df.shape)
books_df.head()

## Result

The notebook creates three files:

- `bengaluru_weather.csv`
- `bengaluru_air_quality.csv`
- `books_data.csv`

These files are ready for the next steps in the data-science workflow, such as cleaning, exploratory analysis, and visualization.

In [ ]:
files = {
    "Weather": weather_file,
    "Air quality": air_file,
    "Books": books_file,
}

for label, path in files.items():
    size_kb = path.stat().st_size / 1024
    print(f"{label:12s} -> {path.name:30s} {size_kb:8.1f} KB")

print("\nRows collected:")
print("Weather:", len(weather_df))
print("Air quality:", len(air_quality_df))
print("Books:", len(books_df))

## Conclusion

Weather and air-quality readings were collected for five Bengaluru locations using Open-Meteo APIs, and book information was collected from a paginated practice website using BeautifulSoup. All three datasets are stored in CSV format for further analysis.